# 🤖 Real-Time Song Type Classifier

**Pipeline**: `Cluster labels → Train SVM/RF → Real-time inference`

```
Stage 1 (offline, done once):
  Feature matrix X [N×53]  +  song-type labels y [N]
      ↓  train
  SongTypeClassifier (SVM or RandomForest)
      ↓  save
  results/song_classifier.pkl

Stage 2 (real-time, every 2.5s):
  Audio window
      ↓  FeatureExtractor
  53-dim vector
      ↓  SongTypeClassifier.predict()   [<10ms]
  "Calling Song" / "Aggressive Song" / ...
      ↓  BehaviorMonitor
  Alert ⚠️ / ✅
```

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pickle
import json
import time
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from cricket_perception.classifier import SongTypeClassifier, auto_label_clusters, SONG_TYPES

plt.rcParams.update({
    'figure.facecolor': '#0f0f1a', 'axes.facecolor': '#0f0f1a',
    'axes.edgecolor': '#333355', 'axes.labelcolor': '#ccccee',
    'text.color': '#ccccee', 'xtick.color': '#aaaacc', 'ytick.color': '#aaaacc',
    'grid.color': '#1e1e3a', 'font.size': 10,
})

RESULTS = Path('../results')
print('✅ Imports OK')

## 1️⃣ โหลด Features + สร้าง Labels

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
df   = pd.read_csv(RESULTS / 'segments_with_clusters.csv')
pipe = pickle.load(open(RESULTS / 'cluster_pipeline.pkl', 'rb'))
X    = np.load('../features_cache/features_X.npy')
labels = pipe.labels_  # HDBSCAN cluster labels

print(f'Feature matrix : {X.shape}')
print(f'Cluster labels : {labels.shape}  (unique: {len(np.unique(labels))})')

# ── Check if manual labels exist (from notebook 05) ───────────────────────────
MANUAL_LABEL_FILE = RESULTS / 'cluster_manual_labels.json'

if MANUAL_LABEL_FILE.exists():
    manual_labels = {int(k): v for k, v in json.loads(MANUAL_LABEL_FILE.read_text()).items()}
    print(f'\n✅ Manual labels found: {len(manual_labels)} clusters labeled')
    
    # Build song_type per segment from manual labels
    def map_manual_label(song_type_str):
        """Normalise emoji labels from notebook 05 to canonical form."""
        if 'Aggressive' in song_type_str: return 'Aggressive Song'
        if 'Calling' in song_type_str:    return 'Calling Song'
        if 'Courtship' in song_type_str:  return 'Courtship/Low Song'
        if 'Quiet' in song_type_str:      return 'Quiet/Background'
        return 'Noise'
    
    y_all = np.full(len(labels), 'Noise', dtype=object)
    for cid, info in manual_labels.items():
        mask = labels == cid
        y_all[mask] = map_manual_label(info['song_type'])
    y_all[labels == -1] = 'Noise'
    LABEL_SOURCE = 'manual'

else:
    print('\n⚠️  No manual labels found — using auto-labeling (rule-based heuristics)')
    print('   Run notebook 05 to create manual labels for better accuracy.')
    y_all = auto_label_clusters(X, labels)
    LABEL_SOURCE = 'auto'

print(f'\nLabel source: {LABEL_SOURCE}')
unique, counts = np.unique(y_all, return_counts=True)
for lbl, cnt in sorted(zip(unique, counts), key=lambda x: -x[1]):
    print(f'  {lbl:<25s}: {cnt:4d} segments ({cnt/len(y_all):.1%})')

## 2️⃣ Train/Test Split + Train Classifier

In [ ]:
# ── Exclude noise from training (optional — noise is usually obvious) ─────────
EXCLUDE_NOISE = True

if EXCLUDE_NOISE:
    valid_mask = y_all != 'Noise'
    X_use, y_use = X[valid_mask], y_all[valid_mask]
    print(f'Excluding Noise: {valid_mask.sum()} / {len(y_all)} segments used')
else:
    X_use, y_use = X, y_all
    print(f'Including Noise: {len(y_all)} segments used')

# ── Stratified train/test split ───────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_use, y_use,
    test_size=0.2,
    stratify=y_use,
    random_state=42,
)

print(f'\nTrain: {len(X_train)} samples')
print(f'Test : {len(X_test)} samples')
print()
print('Train class distribution:')
for lbl, cnt in zip(*np.unique(y_train, return_counts=True)):
    print(f'  {lbl:<25s}: {cnt}')

In [ ]:
# ── Train SVM classifier ──────────────────────────────────────────────────────
print('Training SVM (RBF kernel)...')
clf_svm = SongTypeClassifier(backend='svm', svm_C=10.0, svm_gamma='scale')
metrics_svm = clf_svm.train(X_train, y_train, cv_folds=5)

print(f'\n✅ SVM Training complete:')
print(f'   Train time      : {metrics_svm["train_time_s"]:.2f}s')
print(f'   CV Accuracy     : {metrics_svm.get("accuracy_mean", "N/A")}')
print(f'   CV Std          : ±{metrics_svm.get("accuracy_std", "N/A")}')
print(f'   Classes         : {metrics_svm["classes"]}')

# ── Train RF classifier ───────────────────────────────────────────────────────
print('\nTraining RandomForest...')
clf_rf = SongTypeClassifier(backend='rf', rf_n_estimators=200)
metrics_rf = clf_rf.train(X_train, y_train, cv_folds=5)

print(f'\n✅ RF Training complete:')
print(f'   Train time      : {metrics_rf["train_time_s"]:.2f}s')
print(f'   CV Accuracy     : {metrics_rf.get("accuracy_mean", "N/A")}')
print(f'   CV Std          : ±{metrics_rf.get("accuracy_std", "N/A")}')

## 3️⃣ Evaluate on Test Set

In [ ]:
# ── Evaluate both classifiers ─────────────────────────────────────────────────
eval_svm = clf_svm.evaluate(X_test, y_test)
eval_rf  = clf_rf.evaluate(X_test, y_test)

print('┌──────────────────────────────────────────────────┐')
print('│            Test Set Evaluation Results           │')
print('├──────────────────────────────────────────────────┤')
print(f'│  SVM   Accuracy         : {eval_svm["accuracy"]:.4f}                  │')
print(f'│  SVM   Balanced Acc     : {eval_svm["balanced_accuracy"]:.4f}                  │')
print(f'│  RF    Accuracy         : {eval_rf["accuracy"]:.4f}                  │')
print(f'│  RF    Balanced Acc     : {eval_rf["balanced_accuracy"]:.4f}                  │')
print('└──────────────────────────────────────────────────┘')

# Pick best model
best_clf  = clf_svm if eval_svm['balanced_accuracy'] >= eval_rf['balanced_accuracy'] else clf_rf
best_eval = eval_svm if eval_svm['balanced_accuracy'] >= eval_rf['balanced_accuracy'] else eval_rf
best_name = 'SVM' if best_clf is clf_svm else 'RF'
print(f'\n🏆 Best model: {best_name} (balanced accuracy = {best_eval["balanced_accuracy"]:.4f})')

In [ ]:
# ── Confusion Matrix Comparison ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Confusion Matrices: SVM vs RandomForest (Test Set)', fontsize=13, color='white')

classes = sorted(set(y_test.tolist()))

for ax, (clf_name, eval_result) in zip(axes, [
    ('SVM (RBF)', eval_svm),
    ('Random Forest', eval_rf),
]):
    cm = confusion_matrix(eval_result['y_true'], eval_result['y_pred'],
                          labels=classes, normalize='true')
    
    sns.heatmap(
        cm, annot=True, fmt='.2f',
        xticklabels=[c[:12] for c in classes],
        yticklabels=[c[:12] for c in classes],
        cmap='YlOrRd',
        vmin=0, vmax=1,
        linewidths=0.5, linecolor='#1a1a2e',
        ax=ax,
        annot_kws={'size': 10, 'color': 'black'},
    )
    ax.set_title(
        f'{clf_name}\nAcc={eval_result["accuracy"]:.3f} | Balanced={eval_result["balanced_accuracy"]:.3f}',
        color='white', fontsize=11, pad=10
    )
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('True', fontsize=10)
    ax.tick_params(axis='x', rotation=30, colors='#aaaacc', labelsize=9)
    ax.tick_params(axis='y', rotation=0,  colors='#aaaacc', labelsize=9)

plt.tight_layout()
fig.savefig(RESULTS / '07_confusion_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print('💾 Saved → results/07_confusion_matrix.png')

In [ ]:
# ── RF Feature Importance ─────────────────────────────────────────────────────
importances = clf_rf.feature_importance()

# Feature names (based on README dim breakdown)
feature_names = (
    [f'MFCC_mean_{i}' for i in range(13)] +
    [f'MFCC_std_{i}'  for i in range(13)] +
    ['SC_centroid_m', 'SC_centroid_s', 'SC_bw_m', 'SC_bw_s',
     'SC_rolloff_m', 'SC_rolloff_s', 'SC_entropy_m', 'SC_entropy_s'] +
    [f'Chroma_{i}' for i in range(12)] + ['ZCR_mean', 'ZCR_std'] +
    ['RMS_mean', 'RMS_std', 'RMS_min', 'RMS_max'] +
    ['ACI']
)

imp_series = pd.Series(importances, index=feature_names).sort_values(ascending=True)
top20 = imp_series.tail(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#ff5252' if 'MFCC' in n else
          '#7c85ff' if 'SC_' in n else
          '#69f0ae' if 'Chroma' in n or 'ZCR' in n else
          '#ffd740' if 'RMS' in n else
          '#ff8a65'
          for n in top20.index]

top20.plot(kind='barh', ax=ax, color=colors, alpha=0.85, edgecolor='none')
ax.set_title('Top 20 Feature Importances (RandomForest)', color='white', fontsize=12)
ax.set_xlabel('Importance')
ax.grid(True, axis='x', alpha=0.2)

# Legend
import matplotlib.patches as mpatches
legend_items = [
    mpatches.Patch(color='#ff5252', label='MFCC'),
    mpatches.Patch(color='#7c85ff', label='Spectral'),
    mpatches.Patch(color='#69f0ae', label='Chroma/ZCR'),
    mpatches.Patch(color='#ffd740', label='RMS'),
    mpatches.Patch(color='#ff8a65', label='ACI'),
]
ax.legend(handles=legend_items, fontsize=9, loc='lower right')

plt.tight_layout()
fig.savefig(RESULTS / '07_feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print('💾 Saved → results/07_feature_importance.png')

## 4️⃣ Inference Speed Benchmark

In [ ]:
# ── Inference latency benchmark ───────────────────────────────────────────────
N_BENCH = 1000
x_single = X_test[0]

# Warm up
for _ in range(10):
    clf_svm.predict(x_single)
    clf_rf.predict(x_single)

# SVM
t0 = time.perf_counter()
for _ in range(N_BENCH):
    clf_svm.predict(x_single)
svm_ms = (time.perf_counter() - t0) / N_BENCH * 1000

# RF
t0 = time.perf_counter()
for _ in range(N_BENCH):
    clf_rf.predict(x_single)
rf_ms = (time.perf_counter() - t0) / N_BENCH * 1000

print('⏱️  Single inference latency (mean over {:,} calls):'.format(N_BENCH))
print(f'   SVM : {svm_ms:.3f} ms/sample')
print(f'   RF  : {rf_ms:.3f} ms/sample')
print()
print('📊 Context (window = 2.5s):')
print(f'   Feature extraction : ~40-60ms')
print(f'   SVM inference      : {svm_ms:.3f}ms')
print(f'   Total pipeline     : ~{50 + svm_ms:.0f}ms  (vs 2500ms window = {(50+svm_ms)/2500:.1%} overhead)')
print()
print('✅ Well within real-time budget — can run on Raspberry Pi 4')

## 5️⃣ Save Best Model

In [ ]:
# ── Save best model ───────────────────────────────────────────────────────────
clf_path = RESULTS / 'song_classifier.pkl'
best_clf.save(clf_path)
print(f'✅ Saved best model ({best_name}) → {clf_path}')
print(f'   Balanced accuracy: {best_eval["balanced_accuracy"]:.4f}')
print(f'   Classes: {best_clf.classes}')
print()

# Save metadata
meta = {
    'backend': best_name,
    'label_source': LABEL_SOURCE,
    'n_train': int(len(X_train)),
    'n_test': int(len(X_test)),
    'accuracy': float(best_eval['accuracy']),
    'balanced_accuracy': float(best_eval['balanced_accuracy']),
    'classes': best_clf.classes,
    'feature_dim': int(X.shape[1]),
    'cv_accuracy': metrics_svm.get('accuracy_mean') if best_clf is clf_svm else metrics_rf.get('accuracy_mean'),
}
(RESULTS / 'song_classifier_meta.json').write_text(json.dumps(meta, indent=2))
print('💾 Saved metadata → results/song_classifier_meta.json')

## 6️⃣ Real-Time Stream Demo

ทดสอบ `RealTimeMonitor` บน WAV file จาก dataset
(mode เดียวกับที่จะใช้บน edge device — เปลี่ยนแค่ `from_file` → `from_mic`)

In [ ]:
# ── Load calibration values ───────────────────────────────────────────────────
calib_file = RESULTS / '06_calibrated_thresholds.json'
if calib_file.exists():
    calib = json.loads(calib_file.read_text())
    RMS_BASELINE = calib['rms_baseline']
    ACI_BASELINE = calib['aci_baseline']
    AGG_THRESHOLD = calib['aggressive_threshold']
    RMS_LOW = calib['rms_low_threshold']
    ACI_LOW = calib['aci_low_threshold']
    print('✅ Loaded calibrated thresholds from notebook 06')
else:
    # Fallback: use dataset percentiles
    X_nn = X[pipe.labels_ != -1]
    RMS_BASELINE  = float(np.percentile(X_nn[:, 48:52].mean(axis=1), 60))
    ACI_BASELINE  = float(np.percentile(X_nn[:, 52], 60))
    AGG_THRESHOLD = 0.35
    RMS_LOW, ACI_LOW = 0.40, 0.50
    print('⚠️ Using dataset-derived baselines (run notebook 06 for calibrated values)')

print(f'   RMS baseline      : {RMS_BASELINE:.5f}')
print(f'   ACI baseline      : {ACI_BASELINE:.2f}')
print(f'   Hunger threshold  : {AGG_THRESHOLD:.2f}')
print(f'   Mortality RMS/ACI : {RMS_LOW:.2f} / {ACI_LOW:.2f}')

In [ ]:
# ── Create RealTimeMonitor ────────────────────────────────────────────────────
from cricket_perception.realtime import RealTimeMonitor

rt_monitor = RealTimeMonitor(
    classifier_path      = clf_path,
    rms_baseline         = RMS_BASELINE,
    aci_baseline         = ACI_BASELINE,
    aggressive_threshold = AGG_THRESHOLD,
    rms_low_threshold    = RMS_LOW,
    aci_low_threshold    = ACI_LOW,
    window_s             = 2.5,
    hop_s                = 1.0,
    temperature_c        = 28.0,
)
print('✅ RealTimeMonitor ready')

In [ ]:
# ── Stream a real WAV file from dataset ───────────────────────────────────────
WAV_ROOT = '../dataset/insectset32/Orthoptera/Orthoptera'

# Pick a Gryllus campestris file (closest to farm cricket)
import glob
gryllus_files = sorted(glob.glob(f'{WAV_ROOT}/Grylluscampestris/*.wav'))
if not gryllus_files:
    gryllus_files = sorted(glob.glob(f'{WAV_ROOT}/**/*.wav', recursive=True))

test_wav = gryllus_files[0]
print(f'📂 Streaming: {test_wav}')
print()
print(f'{"Time":<10} {"Song Type":<25} {"Conf":<8} {"RMS":<10} {"ACI":<8} Alert')
print('─' * 75)

stream_results = []
for result in rt_monitor.stream_file(test_wav, verbose=False):
    t = f"{result.window_start_s:.1f}s"
    alert_str = '  '.join(result.alerts) if result.alerts else '✅'
    print(f'{t:<10} {result.song_type:<25} {result.confidence:.0%}     '
          f'{result.rms:.5f}   {result.aci:.0f}  {alert_str}')
    stream_results.append(result)

print(f'\n📊 Streamed {len(stream_results)} windows')
types = [r.song_type for r in stream_results]
for t, c in pd.Series(types).value_counts().items():
    print(f'   {t:<25}: {c}')

## 7️⃣ Visualise Stream Results

In [ ]:
# ── Plot stream timeline ───────────────────────────────────────────────────────
if not stream_results:
    print('No stream results to plot')
else:
    import matplotlib.patches as mpatches

    TYPE_COLORS = {
        'Aggressive Song':    '#ff5252',
        'Calling Song':       '#69f0ae',
        'Courtship/Low Song': '#40c4ff',
        'Quiet/Background':   '#aaaacc',
        'Noise':              '#444466',
    }

    sr_df = pd.DataFrame([
        {
            't': r.window_start_s,
            'song_type': r.song_type,
            'confidence': r.confidence,
            'rms': r.rms,
            'aci': r.aci,
            'hungry': r.hungry,
            'mortality': r.mortality_risk,
        }
        for r in stream_results
    ])

    fig = plt.figure(figsize=(16, 10))
    gs  = gridspec.GridSpec(3, 1, figure=fig, hspace=0.45)
    fig.suptitle(f'Real-Time Inference Stream\n{Path(test_wav).name}',
                 fontsize=13, color='white')

    # Panel 1: Song type timeline (coloured bars)
    ax1 = fig.add_subplot(gs[0])
    for _, row in sr_df.iterrows():
        color = TYPE_COLORS.get(row['song_type'], '#777799')
        ax1.barh(0, 1.0, left=row['t'], height=0.6, color=color, alpha=0.85, edgecolor='none')
    ax1.set_xlim(sr_df['t'].min(), sr_df['t'].max() + 2.5)
    ax1.set_yticks([])
    ax1.set_xlabel('Time (s)')
    ax1.set_title('🎵 Song Type Timeline', color='white', fontsize=11)

    legend_items = [mpatches.Patch(color=c, label=t) for t, c in TYPE_COLORS.items()]
    ax1.legend(handles=legend_items, loc='upper right', fontsize=8, framealpha=0.3, ncol=3)
    ax1.grid(True, axis='x', alpha=0.15)

    # Panel 2: Confidence + RMS
    ax2 = fig.add_subplot(gs[1])
    ax2b = ax2.twinx()

    bar_colors = [TYPE_COLORS.get(t, '#777799') for t in sr_df['song_type']]
    ax2.bar(sr_df['t'], sr_df['confidence'], width=0.8, color=bar_colors, alpha=0.7, label='Confidence')
    ax2b.plot(sr_df['t'], sr_df['rms'], color='white', lw=1.5, alpha=0.8, label='RMS')

    ax2.axhline(0.5, color='#ffd740', ls=':', lw=1, alpha=0.7, label='50% confidence')
    ax2.set_ylabel('Classification Confidence', fontsize=9, color='white')
    ax2b.set_ylabel('RMS Energy', fontsize=9, color='#aaaacc')
    ax2.set_ylim(0, 1.1)
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax2.set_title('Confidence & RMS Energy', color='white', fontsize=11)
    ax2.grid(True, alpha=0.15)
    ax2.set_xlabel('Time (s)')

    lines1, labs1 = ax2.get_legend_handles_labels()
    lines2, labs2 = ax2b.get_legend_handles_labels()
    ax2.legend(lines1 + lines2, labs1 + labs2, fontsize=8, loc='upper right')

    # Panel 3: Alert markers
    ax3 = fig.add_subplot(gs[2])
    ax3.fill_between(sr_df['t'], sr_df['aci'], alpha=0.3, color='#7c85ff')
    ax3.plot(sr_df['t'], sr_df['aci'], color='#7c85ff', lw=1.5, label='ACI')
    ax3.axhline(ACI_BASELINE * ACI_LOW, color='#ff5252', ls=':', lw=1.5, label=f'Mortality threshold ({ACI_LOW:.0%} baseline)')

    for _, row in sr_df[sr_df['hungry']].iterrows():
        ax3.axvspan(row['t'], row['t'] + 2.5, color='#ffd740', alpha=0.15)
    for _, row in sr_df[sr_df['mortality']].iterrows():
        ax3.axvspan(row['t'], row['t'] + 2.5, color='#ff5252', alpha=0.2)

    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('ACI')
    ax3.set_title('ACI + Alert Regions', color='white', fontsize=11)
    ax3.legend(fontsize=8)
    ax3.grid(True, alpha=0.15)

    for ax in [ax1, ax2, ax3]:
        ax.set_facecolor('#0f0f1a')
        for spine in ax.spines.values():
            spine.set_edgecolor('#333355')

    plt.tight_layout()
    fig.savefig(RESULTS / '07_realtime_stream.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
    plt.show()
    print('💾 Saved → results/07_realtime_stream.png')

## 8️⃣ Microphone Mode — วิธีใช้งานจริง

> ⚠️ ต้องติดตั้ง libportaudio2 ก่อน:
> ```bash
> sudo apt-get install libportaudio2
> pip install sounddevice
> ```

In [ ]:
# ── Mic mode template (run this in a Python script, not notebook) ─────────────
MIC_CODE = '''
import sys, json
sys.path.insert(0, "src")
from pathlib import Path
from cricket_perception.realtime import RealTimeMonitor

# Load calibrated thresholds
calib = json.loads(Path("results/06_calibrated_thresholds.json").read_text())

monitor = RealTimeMonitor.from_mic(
    classifier_path      = "results/song_classifier.pkl",
    rms_baseline         = calib["rms_baseline"],
    aci_baseline         = calib["aci_baseline"],
    aggressive_threshold = calib["aggressive_threshold"],
    rms_low_threshold    = calib["rms_low_threshold"],
    aci_low_threshold    = calib["aci_low_threshold"],
    window_s             = 2.5,
    hop_s                = 1.0,
    temperature_c        = 28.0,   # <- ส่งมาจาก sensor
)

def on_alert(result):
    print(f"🚨 ALERT at {result.timestamp}: {result.alerts}")
    # ส่ง LINE Notify / MQTT / webhook ที่นี่

print("🎙️  Starting real-time monitoring... (Ctrl+C to stop)")
monitor.start_mic(
    device    = None,       # None = default mic
    on_alert  = on_alert,
    verbose   = True,
)

try:
    import time
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    results = monitor.stop_mic()
    print(f"\nStopped. Processed {len(results)} windows.")
'''

# Save as runnable script
script_path = Path('../scripts/realtime_monitor.py')
script_path.write_text(MIC_CODE.strip())
print(f'✅ Saved runnable mic script → {script_path}')
print()
print('Run with:')
print('  source .venv/bin/activate')
print('  python scripts/realtime_monitor.py')
print()
print('Code preview:')
print(MIC_CODE)

## 9️⃣ สรุป Full Pipeline

```
Offline Training (ทำครั้งเดียว):
┌─────────────────────────────────────────────────────────────┐
│  01_explore_dataset.ipynb     → เข้าใจ data              │
│  02_feature_extraction.ipynb  → X [759 × 53]              │
│  03_clustering.ipynb          → UMAP + HDBSCAN → clusters │
│  04_behavior_analysis.ipynb   → behavior alerts sim       │
│  05_cluster_audio_labeling.ipynb → manual song-type labels│
│  06_baseline_calibration.ipynb → RMS/ACI baselines        │
│  07_realtime_classifier.ipynb  → SVM/RF trained ✅        │
└─────────────────────────────────────────────────────────────┘

Real-Time Inference (ทำงานต่อเนื่อง):
┌─────────────────────────────────────────────────────────────┐
│  Microphone (2.5s window)                                  │
│      ↓  FeatureExtractor  (~40ms)                          │
│  53-dim vector                                             │
│      ↓  SongTypeClassifier.predict()  (<1ms)               │
│  "Calling Song" | "Aggressive Song" | ...                  │
│      ↓  BehaviorMonitor.analyze()     (<1ms)               │
│  Alert ⚠️ / ✅  → LINE / MQTT / Dashboard                 │
└─────────────────────────────────────────────────────────────┘

Total inference latency: ~50ms per 2.5s window = 2% overhead
Runs on: Raspberry Pi 4, laptop, any CPU device
```

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('═' * 60)
print('  🦗 REALTIME CLASSIFIER — TRAINING SUMMARY')
print('═' * 60)
print(f'  Best model       : {best_name}')
print(f'  Label source     : {LABEL_SOURCE}')
print(f'  Train size       : {len(X_train)}')
print(f'  Test size        : {len(X_test)}')
print(f'  Accuracy (test)  : {best_eval["accuracy"]:.4f}')
print(f'  Balanced Acc     : {best_eval["balanced_accuracy"]:.4f}')
print(f'  Inference speed  : {svm_ms:.2f}ms/sample')
print(f'  Classes          : {best_clf.classes}')
print()
print('  Saved files:')
for f in [
    'song_classifier.pkl',
    'song_classifier_meta.json',
    '07_confusion_matrix.png',
    '07_feature_importance.png',
    '07_realtime_stream.png',
]:
    print(f'    ✅ results/{f}')
print('  Script:')
print(f'    ✅ scripts/realtime_monitor.py')
print('═' * 60)